# Clean Hopkins Data

In [ ]:
import sys
import os

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
import pandas as pd
import numpy as np

DATA_DIR = BASE_PATH / "data" / "raw"
# Import Data
df = pd.read_excel(DATA_DIR / "hopkins_ORN.xlsx")

In [ ]:
rename_dict = {
    ## Pre-Op chars
    "AGE": "AGE",
    "BMI (kg/m^2)": "BMI",
    "SEX (female=0, male=1)": "SEX",
    "DIABETES (no=0, yes=1)": "Diabetes",
    "ASA SCORE (ASA 1 = 1, ASA 2 = 2, ASA 3 = 3, ASA 4 = 4)": "ASA",
    "PRIOR SURGERY IN SAME PLACE (no=0, yes=1)": "PRIOREX",
    "PREOP CT  (no=0, yes=1)": "PRECT",
    "PREOP RT  (no=0, yes=1)": "PRERT",
    ## Disease Chars
    "RECURRENT CANCER (no=0, yes=1)": "RECUR",
    "TUMOR SITE (mouth floor = 1, buccal = 2, retromolar = 3, gum = 4, tongue = 5, lip  = 6, palate = 7)": "SITE",
    "TUMOR SIZE (T1 = 1, T2 = 2, T3 = 3, T4 = 4)": "SIZE",
    "LYMPH NODE INVOLVEMENT  (no=0, yes=1)": "LYMPH",
    "STAGE (stage 1=0, stage 2/3=1, stage 4a/b=2)": "STAGE",
    "DEFECT TYPE (intraoral = 0, composite = 1)": "DEFECT",
    "SECONDARY PRIMARY CANCER (no=0, yes=1)": "SECONDPRIMARY",
    ## Surg Chars
    "FIBULA LENGTH (cm)": "LENGTH",
    "JEWER BOYD CLASSIFICATION (L=0, LC=1, LCL=2)": "JEWER",
    "OSTEOTOMIES (0=0, 1=1, 2+=2)": "OSTEOTOMY",
    "PLATE TYPE (R = reconstruction, M = miniplates, P = preformed)": "PLATE",
    "FLAP TYPE (osteocutaneous = 0, chimeric = 1)": "FLAP",
    "INTRAOP BLOOD TXN  (no=0, yes=1)": "TRANSFUS",
    "ISCHEMIC TIME (minutes)": "ISCHEMICTIME",
    "OP TIME (min)": "OPTIME",
    ## Immediate post-Op Chars
    "REOPERATION  (no=0, yes=1)": "REOP",
    "LOHS (days)": "LOHS",
    "POSTOP RT  (no=0, yes=1)": "POSTRT",
    "POSTOP CT  (no=0, yes=1)": "POSTCT",
    "WOUND INFECTION  (no=0, yes=1)": "WOUNDINF",
    "POSTOP HGB (g/dL)": "HGB",
    "POSTOP ALBUMIN (g/dL)": "ALB",
    ## Long term post-op
    "PLATE EXPOSURE  (no=0, yes=1)": "EXPOSURE",
    "PLATE EXPOSURE TREATED WITH MED  (no=0, yes=1)": "MEDUSED",
    "PLATE EXPOSURE Tx NO MED (none = 0, plate removal = 1, flap = 2)": "SURGUSED",
    "TIME TO PLATE EXPOSURE (years)": "PLATETIME",
    "TIME FROM FFF TO MOST RECENT FU": "FOLLOWTIME",
    ## Target
    "ORN  (no=0, yes=1)": "ORN",
}
df_rename = df.rename(columns=rename_dict)
df_sub = df_rename[rename_dict.values()].copy()
# Subset only RT patients
df_sub = df_sub[(df_sub["POSTRT"] == 1) | (df_sub["PRERT"] == 1)]
df_sub.shape

Deal with NAs

- OSTEOTOMY --> NA means NO (0)

- MEDUSED/SURGUSED/PLATETIME --> NA means no exposure
    - Exposure == 0 for all of these

In [ ]:
# Osteotomy
df_sub["OSTEOTOMY"] = df_sub["OSTEOTOMY"].fillna(0)
##Exposures
df_sub["MEDUSED"] = df_sub["MEDUSED"].fillna(0)
df_sub["SURGUSED"] = df_sub["SURGUSED"].fillna(0)
df_sub["PLATETIME"] = df_sub["PLATETIME"].fillna(0)

Reformat Exposure vars

In [ ]:
df_clean = df_sub.copy()
## Time to plate exposure
df_clean["PLATETIME"] = pd.cut(
    df_clean["PLATETIME"],
    bins=[0, 1, 2, float("inf")],
    labels=["0to1", "1to2", "2plus"],
    right=False,  # means [0, 1) is "Early", [1, 2) is "Intermediate", [2, inf) is "Late"
)
df_clean["PLATETIME"] = np.where(
    df_clean["EXPOSURE"] == 0, "NoExposure", df_clean["PLATETIME"]
)

## Medication Used for plate
df_clean["MEDUSED"] = np.select(
    [
        df_clean["EXPOSURE"] == 0,
        (df_clean["EXPOSURE"] == 1) & (df_clean["MEDUSED"] == 1),  # Exp + med used
        (df_clean["EXPOSURE"] == 1) & (df_clean["MEDUSED"] == 0),  # Exp + no med used
    ],
    ["NoExposure", "Yes", "No"],
    default="Unknown",
)
## Surgical Intervention Used for plate
df_clean["SURGUSED"] = np.select(
    [
        df_clean["EXPOSURE"] == 0,
        (df_clean["EXPOSURE"] == 1) & (df_clean["SURGUSED"] == 0),  # None
        (df_clean["EXPOSURE"] == 1) & (df_clean["SURGUSED"] == 1),  # plate
        (df_clean["EXPOSURE"] == 1) & (df_clean["SURGUSED"] == 2),  # flap
    ],
    ["NoExposure", "None", "Plate", "Flap"],
    default="Unknown",
)
df_clean["PLATETIME"].value_counts()

Rework other features

In [ ]:
##ASA make binary
df_clean["ASA"] = df_clean["ASA"].replace({3: "ASA3", 2: "ASA1/2"})
df_clean["ASA"] = df_clean["ASA"].replace({"ASA3": 1, "ASA1/2": 0})
df_clean["ASA"].value_counts()
## SITE combined 1 instance of 7 (palate) w/ 4 (gum)
df_clean["SITE"] = df_clean["SITE"].replace({7: 4})
## SIZE make binary
df_clean["SIZE"] = df_clean["SIZE"].replace({1: "T1", 2: "T2", 3: "T3", 4: "T4"})
df_clean["SIZE"] = df_clean["SIZE"].replace({"T1": 0, "T2": 0, "T3": 1, "T4": 1})
## STAGE --> 1= stage1, 2=stage2/3, 3=stage4
df_clean["STAGE"] = df_clean["STAGE"].replace({0: "Stage1", 1: "Stage2/3", 2: "Stage4"})
df_clean["STAGE"] = df_clean["STAGE"].replace({"Stage1": 1, "Stage2/3": 2, "Stage4": 3})
## JEWER -->  0=C, 1=L, 3=LC, 5=LCL)
df_clean["JEWER"] = df_clean["JEWER"].replace({0: "L", 1: "LC", 2: "LCL"})
df_clean["JEWER"] = df_clean["JEWER"].replace({"L": 1, "LC": 3, "LCL": 5})
## PLATE -->  0=mini plate 1=reconstruction plate 2=preformed plate
df_clean["PLATE"] = df_clean["PLATE"].replace({"R": 1, "M": 0, "P": 2})

General Cleaning

In [ ]:
## Surgical Intervention Used for plate
df_clean["RADTIME"] = np.select(  # note that no instances of neither
    [
        (df_clean["POSTRT"] == 1) & (df_clean["PRERT"] == 0),  # Just post
        (df_clean["POSTRT"] == 0) & (df_clean["PRERT"] == 1),  # Just pre
        (df_clean["POSTRT"] == 1) & (df_clean["PRERT"] == 1),  # Both
    ],
    ["Post", "Pre", "Both"],
    default="Unknown",
)
df_clean = df_clean.drop(["PRERT", "POSTRT"], axis=1)
df_clean["RADTIME"].value_counts()

In [ ]:
for col in df_sub.columns:
    n_na = df_sub[col].isna().sum()
    if n_na > 0:
        print(col)
        print(n_na)

Export

In [ ]:
save_path = DATA_DIR / "Cleaned_hopkins.parquet"
if save_path.exists():
    save_path.unlink()
df_clean.to_parquet(save_path)